# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bander03/FlyRank_Intern/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract — five answers in plain words

**1 — One row means:**
One content item on one calendar day: the GSC and GA4 signals Google reported for that URL, for that client, on that specific date. Grain: `report_date × client_hash_id × content_hash_id`.

**2 — Tables I'll use:**
- `fact_content_daily_performance` (partitioned by `month`) — daily impressions, clicks, position, GA4 sessions. Primary source.
- `dim_content` — content metadata (type, word count) for joins. Context only.
- `dim_clients` — `gsc_data_start` and `ga4_data_start` to filter out incomplete history before building features.

**3 — Time window:**
Development and verification run on **`month = 2026-03`** (a mid-panel month). Features are computed from days 1–15 of March; the label is derived from days 16–31. The sealed test month (June 2026 / the `_sample` table) is untouched.

**4 — What I'd predict (label proxy):**
Whether a content item's impressions dropped by more than 20 % in the second half of March compared to the first half:
`is_declining = (imp_last16 < 0.8 × imp_first15)`.
This is a directional proxy for content that may need refreshing, not a causal determination.

**5 — One deliberate exclusion:**
`ga4_data_available = FALSE` rows. Before a client's GA4 start date the pipeline zero-fills every GA4 column and marks this flag FALSE. Those zeros are not "no engagement" — they are missing data. Including them would inject a client-age signal into features silently.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

| Field | Bucket | Notes |
|---|---|---|
| `report_date` | Context | Window definition and ordering only |
| `client_hash_id` | Context | Client-holdout splits only; never a feature |
| `content_hash_id` | Context | Join key only |
| `gsc_impressions` | **Feature** (prev window) | Aggregate over days 1–15; knowable before label period |
| `gsc_clicks` | **Feature** (prev window) | Same window as impressions; reported together by GSC |
| `gsc_avg_position` | **Feature** | Position is reported alongside impressions; no future information |
| `ctr_first15` (derived) | **Feature** | Computed as `clk_first15 / imp_first15`; no stored `gsc_ctr` column |
| `ga4_sessions` | **Feature** (prev window, IS TRUE only) | Safe once `ga4_data_available IS TRUE` filter is applied |
| `ga4_data_available` | Context | Filter flag — `FALSE` rows are excluded, not zeroed |
| `imp_last16` (derived) | **Label source** | Used to compute `is_declining` — never a model feature |
| `content_type` (dim_content) | **Feature** | Set at content creation; knowable at any decision point |
| `word_count` (dim_content) | **Feature** | Static metadata; knowable before any outcome |
| `month` (partition key) | Context | Query efficiency only |
| `ga4_engaged_sessions` | Excluded | Correlated with label in zero-filled rows; redundant with sessions after IS TRUE filter |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import os, getpass
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# ── Connect to warehouse ──────────────────────────────────────────────────────
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF READ token (hf_...): ")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL       = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR  = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONT  = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected. Development partition: month=2026-03")
print("-" * 60)

# ════════════════════════════════════════════════════════════════
# QUERY 1 — Grain: one row = one report_date × client × content?
# ════════════════════════════════════════════════════════════════
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MAR}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Query 1 — Grain check")
print(f"  Duplicate (report_date, client, content) rows found: {len(grain)}")
print("  → 0 confirms: one row = one date × client × content item")
print()

# ════════════════════════════════════════════════════════════════
# QUERY 2 — Row count and date span for month=2026-03
# ════════════════════════════════════════════════════════════════
span = con.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT client_hash_id)    AS distinct_clients,
        COUNT(DISTINCT content_hash_id)   AS distinct_content_items,
        MIN(report_date)                  AS first_date,
        MAX(report_date)                  AS last_date
    FROM {FACT_MAR}
""").df()

print("Query 2 — Row count and date span")
print(span.to_string(index=False))
print()

# ════════════════════════════════════════════════════════════════
# QUERY 3 — Availability: filter ga4_data_available IS TRUE
# ════════════════════════════════════════════════════════════════
avail = con.sql(f"""
    SELECT
        COUNT(*)                                                          AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END)     AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END)     AS ga4_filled_zeros,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
              / COUNT(*), 1)                                              AS pct_available
    FROM {FACT_MAR}
""").df()

print("Query 3 — GA4 availability (IS TRUE filter)")
print(avail.to_string(index=False))
print("  → Only ga4_available_rows rows carry real engagement signal.")
print("  → ga4_filled_zeros rows have pipeline-injected zeros — excluded from GA4 features.")
print()

# ════════════════════════════════════════════════════════════════
# FIVE FEATURES — available when?
# ════════════════════════════════════════════════════════════════
print("Five features — 'knowable at the decision moment because…'")
notes = [
    ("imp_first15",      "impressions summed over days 1–15 of March",
     "feature window (days 1–15) ends before the label window (days 16–31)"),
    ("clk_first15",      "clicks summed over days 1–15 of March",
     "same GSC reporting window as impressions — reported together"),
    ("pos_first15",      "avg GSC position over days 1–15",
     "position is delivered alongside impressions with the same reporting lag"),
    ("ctr_first15",      "computed CTR = clk_first15 / imp_first15 over days 1–15",
     "derived entirely from clk_first15 and imp_first15 — both in the safe window"),
    ("sessions_first15", "GA4 sessions over days 1–15, IS TRUE rows only",
     "GA4 flag filter removes zero-filled rows; remaining sessions predate the label window"),
]
for feat, meaning, reason in notes:
    print(f"  {feat:<18} | {meaning}")
    print(f"  {'':18}   knowable because {reason}")
print()

# ════════════════════════════════════════════════════════════════
# FEATURE FRAME for month=2026-03
# ════════════════════════════════════════════════════════════════
# Note: gsc_ctr is not a stored column — CTR is computed from aggregated clicks/impressions
feat_df = con.sql(f"""
    WITH mar AS (
        SELECT
            client_hash_id,
            content_hash_id,
            -- Safe feature window: first 15 days
            SUM(CASE WHEN DAY(report_date) <= 15 THEN gsc_impressions ELSE 0 END)  AS imp_first15,
            SUM(CASE WHEN DAY(report_date) <= 15 THEN gsc_clicks      ELSE 0 END)  AS clk_first15,
            AVG(CASE WHEN DAY(report_date) <= 15 THEN gsc_avg_position END)         AS pos_first15,
            -- CTR computed from aggregate clicks / impressions (no stored gsc_ctr column)
            SUM(CASE WHEN DAY(report_date) <= 15 THEN gsc_clicks ELSE 0 END) * 1.0
              / NULLIF(SUM(CASE WHEN DAY(report_date) <= 15
                               THEN gsc_impressions ELSE 0 END), 0)                AS ctr_first15,
            SUM(CASE WHEN DAY(report_date) <= 15
                      AND ga4_data_available IS TRUE
                     THEN ga4_sessions ELSE 0 END)                                  AS sessions_first15,
            -- Label window: last 16 days (outcome — not a feature)
            SUM(CASE WHEN DAY(report_date) > 15  THEN gsc_impressions ELSE 0 END)  AS imp_last16
        FROM {FACT_MAR}
        GROUP BY client_hash_id, content_hash_id
        HAVING imp_first15 >= 50   -- require some signal in the feature window
    )
    SELECT * FROM mar
""").df()

print(f"Feature frame built: {len(feat_df):,} content-item rows")
print(feat_df.describe().round(1))
print()

# ════════════════════════════════════════════════════════════════
# LABEL + HONEST QUICK SCORE
# ════════════════════════════════════════════════════════════════
feat_df["is_declining"] = (feat_df["imp_last16"] < 0.8 * feat_df["imp_first15"]).astype(int)
print(f"Label base rate  is_declining=1: {feat_df['is_declining'].mean():.1%}")

feat_cols  = ["imp_first15", "clk_first15", "pos_first15", "ctr_first15", "sessions_first15"]
model_data = feat_df.dropna(subset=feat_cols).reset_index(drop=True)

X = model_data[feat_cols].fillna(0)
y = model_data["is_declining"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf      = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
honest_p = precision_score(y_te, clf.predict(X_te))
print(f"Honest Precision (safe features only): {honest_p:.3f}")
print()

# ════════════════════════════════════════════════════════════════
# THE TRAP — add a label-derived column, watch the score jump
# ════════════════════════════════════════════════════════════════
# imp_last16 IS the label derivation: is_declining = (imp_last16 < 0.8 * imp_first15)
# A model that sees imp_last16 simply memorises the formula.
# Align leaky feature with the SAME train/test indices from the honest split.

X_leaky_tr = X_tr.copy()
X_leaky_tr["imp_last16"] = model_data.loc[X_tr.index, "imp_last16"].values
X_leaky_te = X_te.copy()
X_leaky_te["imp_last16"] = model_data.loc[X_te.index, "imp_last16"].values

clf_leaky = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_leaky_tr, y_tr)
leaky_p   = precision_score(y_te, clf_leaky.predict(X_leaky_te))

print("THE TRAP")
print(f"  Leaky Precision  (imp_last16 included):  {leaky_p:.3f}  ← looks great")
print(f"  Honest Precision (safe features only):   {honest_p:.3f}  ← the real number")
print()
print("imp_last16 is deleted from the feature set.")
print("It defines the label — the model just learned to recompute is_declining directly.")

# imp_last16 dropped — it is the label source
feat_df.drop(columns=["imp_last16"], inplace=True)
print(f"Columns remaining after drop: {list(feat_df.columns)}")

Note: you may need to restart the kernel to use updated packages.


Connected. Development partition: month=2026-03
------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Grain check
  Duplicate (report_date, client, content) rows found: 0
  → 0 confirms: one row = one date × client × content item



Query 2 — Row count and date span
 total_rows  distinct_clients  distinct_content_items first_date  last_date
    9841378                55                  331437 2026-03-01 2026-03-31



...

Query 3 — GA4 availability (IS TRUE filter)
 total_rows  ga4_available_rows  ga4_filled_zeros  pct_available
    9841378            413966.0         6408671.0            4.2
  → Only ga4_available_rows rows carry real engagement signal.
  → ga4_filled_zeros rows have pipeline-injected zeros — excluded from GA4 features.

Five features — 'knowable at the decision moment because…'
  imp_first15        | impressions summed over days 1–15 of March
                       knowable because feature window (days 1–15) ends before the label window (days 16–31)
  clk_first15        | clicks summed over days 1–15 of March
                       knowable because same GSC reporting window as impressions — reported together
  pos_first15        | avg GSC position over days 1–15
                       knowable because position is delivered alongside impressions with the same reporting lag
  ctr_first15        | computed CTR = clk_first15 / imp_first15 over days 1–15
                       knowable bec

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame built: 92,548 content-item rows
       imp_first15  clk_first15  pos_first15  ctr_first15  sessions_first15  \
count      92548.0      92548.0      92548.0      92548.0           92548.0   
mean        1368.9          4.1         13.8          0.0               5.8   
std         3323.2         16.8         14.2          0.0              22.9   
min           50.0          0.0          0.0          0.0               0.0   
25%          143.0          0.0          4.6          0.0               0.0   
50%          406.0          1.0          8.1          0.0               0.0   
75%         1232.0          3.0         18.0          0.0               2.0   
max       161575.0       2395.0        127.6          0.2            1202.0   

       imp_last16  
count     92548.0  
mean       1586.2  
std        4229.5  
min           0.0  
25%         157.0  
50%         454.0  
75%        1460.0  
max      455549.0  

Label base rate  is_declining=1: 28.6%


Honest Precision (safe features only): 0.383



THE TRAP
  Leaky Precision  (imp_last16 included):  0.985  ← looks great
  Honest Precision (safe features only):   0.383  ← the real number

imp_last16 is deleted from the feature set.
It defines the label — the model just learned to recompute is_declining directly.
Columns remaining after drop: ['client_hash_id', 'content_hash_id', 'imp_first15', 'clk_first15', 'pos_first15', 'ctr_first15', 'sessions_first15', 'is_declining']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### One named limitation of this slice

**Unbalanced panel depth makes the March 2026 feature window inconsistent across clients.**

Not every client has GSC data going back to 2025-01-27. A content item's `imp_first15` for March 2026 reflects however long that client has been on the platform — a client who onboarded in February 2026 has only their second month of data, while a client active since January 2025 has 14+ months of accumulated SEO history. The absolute impression volumes are therefore not comparable across clients without normalisation.

**Consequence:** A model trained on raw `imp_first15` implicitly learns client age as a proxy signal. Before scaling to the full panel, features should either be normalised per client (e.g. as a fraction of the client's trailing average) or the training set should be filtered to clients with `gsc_data_start <= 2025-12-01` to guarantee a minimum 3-month history before the development month. Check `dim_clients.gsc_data_start` before extending the feature window.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.